# 🎮 移动游戏用户增长与行为洞察分析

**项目定位：** 基于 Gamelytics 移动游戏真实业务数据（100 万注册玩家、960 万登录记录、40.5 万 A/B 测试用户），构建一套覆盖「用户增长 → 留存 → 商业化 → 玩家画像 → 流失预测 → 实时运营监控」的全链路数据分析项目。

**分析主线（一条完整的数据叙事）：**

1. **数据加载与预处理** — 三张核心业务表的字段字典、缺失值、时间戳清洗与数据质量验证
2. **探索性数据分析 (EDA)** — 注册趋势、UID 完整性、缺失日期热力图、季节性（二月/九月）洞察
3. **指标体系与留存分析** — 每日留存率函数 + 月度群组留存热力图 + 玩家生命周期
4. **A/B 测试与增长量化** — ARPU/CR/ARPPU 三维指标 + 统计检验 + Cohen's d 效应量评分模型
5. **玩家画像与行为聚类** — 登录间隔/生命周期 + K-Means / DBSCAN / GMM 三种聚类
6. **流失预测建模** — 特征工程 + 逻辑回归 / 随机森林 / XGBoost
7. **时间序列与活动趋势** — 全局趋势、个体生命周期曲线、流失前兆、季节性
8. **游戏主题活动评估框架** — Plants & Gardens 限时活动七维指标体系
9. **实时监控预警系统** — 动态风险评分 + 多级预警 + 运营监控仪表板
10. **综合结论与业务建议** — 数据驱动的增长策略输出

**数据来源：** Gamelytics Mobile Analytics Challenge（Kaggle），数据集存放于 `data/` 目录。


## 0. 环境配置与全局样式

首先统一 Python 运行环境：设置标准输出为 UTF-8（规避 Windows GBK 控制台对中文/希腊字母/emoji 的编码报错），并导入本项目所需的全部依赖库、配置中文字体。


In [1]:
# 0.1 统一标准输出编码（Windows 控制台默认 GBK，遇到 χ²/≤/ρ/emoji 会 UnicodeEncodeError）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    sys.stderr.reconfigure(encoding='utf-8', errors='replace')
except (AttributeError, ValueError):
    pass  # Jupyter 内核中 sys.stdout 为 OutStream，无 reconfigure 方法，由内核自行处理 UTF-8

# 0.2 导入所有依赖库
import os
import datetime
import warnings
import numpy as np
import pandas as pd
from typing import List, Tuple

# 数据可视化
import matplotlib
matplotlib.use('Agg')            # 无界面后端，保证 nbconvert 批量执行可用
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick

# 统计分析
from scipy import stats
from scipy.stats import (
    ttest_ind, norm, chi2_contingency, mannwhitneyu,
    levene, pearsonr, spearmanr
)

# 机器学习
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    silhouette_score, davies_bouldin_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

warnings.filterwarnings('ignore')

# 0.3 全局中文&学术样式配置
plt.rcParams["font.family"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["figure.dpi"] = 150
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["lines.linewidth"] = 1.5
plt.rcParams["lines.markersize"] = 4
plt.rcParams["grid.alpha"] = 0.3

# 0.4 输出图目录（供报告引用）
REPORT_DIR = os.getcwd()
FIG_DIR = os.path.join(REPORT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

print('环境配置完成：')
print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}')
print(f'  工作目录: {REPORT_DIR}')
print(f'  图表输出目录: {FIG_DIR}')


环境配置完成：
  Python 3.13.0 | pandas 2.2.3 | numpy 2.4.1
  工作目录: F:\IDE_JetBrains\Python_Pycharm\Resource\Gamelytics
  图表输出目录: F:\IDE_JetBrains\Python_Pycharm\Resource\Gamelytics\figures


## 1. 数据加载与预处理

本项目包含三张核心业务表（分号分隔的 CSV）：

| 数据集 | 记录数 | 字段 | 含义 |
|--------|--------|------|------|
| `reg_data.csv` | 1,000,000 | `reg_ts`(Unix 时间戳), `uid`(用户 ID) | 玩家注册时间 |
| `auth_data.csv` | 9,601,013 | `auth_ts`(Unix 时间戳), `uid`(用户 ID) | 玩家登录/认证时间 |
| `ab_test.csv` | 404,770 | `user_id`, `revenue`(收入), `testgroup`(a/b) | A/B 促销实验结果 |

下面加载数据并做缺失值、数据类型、时间戳转换等清洗工作。


In [2]:
# 1.1 加载三张表（统一使用 data/ 相对路径）
reg_data = pd.read_csv('data/reg_data.csv', delimiter=';')
auth_data = pd.read_csv('data/auth_data.csv', delimiter=';')
ab_test = pd.read_csv('data/ab_test.csv', delimiter=';')

print('=' * 60)
print('数据加载完成')
print('=' * 60)
print(f'reg_data : {reg_data.shape[0]:,} 行 × {reg_data.shape[1]} 列')
print(f'auth_data: {auth_data.shape[0]:,} 行 × {auth_data.shape[1]} 列')
print(f'ab_test  : {ab_test.shape[0]:,} 行 × {ab_test.shape[1]} 列')

# 1.2 缺失值检查
print('\n缺失值检查（各列空值计数）：')
for name, df in [('reg_data', reg_data), ('auth_data', auth_data), ('ab_test', ab_test)]:
    print(f'  {name}: {dict(df.isnull().sum())}')

# 1.3 字段预览
print('\n字段预览：')
print('  reg_data:', list(reg_data.columns), '| dtype:', reg_data.dtypes.to_dict())
print('  auth_data:', list(auth_data.columns), '| dtype:', auth_data.dtypes.to_dict())
print('  ab_test:', list(ab_test.columns), '| dtype:', ab_test.dtypes.to_dict())
print()
print(reg_data.head(3).to_string())
print(auth_data.head(3).to_string())
print(ab_test.head(3).to_string())


数据加载完成
reg_data : 1,000,000 行 × 2 列
auth_data: 9,601,013 行 × 2 列
ab_test  : 404,770 行 × 3 列

缺失值检查（各列空值计数）：
  reg_data: {'reg_ts': np.int64(0), 'uid': np.int64(0)}
  auth_data: {'auth_ts': np.int64(0), 'uid': np.int64(0)}
  ab_test: {'user_id': np.int64(0), 'revenue': np.int64(0), 'testgroup': np.int64(0)}

字段预览：
  reg_data: ['reg_ts', 'uid'] | dtype: {'reg_ts': dtype('int64'), 'uid': dtype('int64')}
  auth_data: ['auth_ts', 'uid'] | dtype: {'auth_ts': dtype('int64'), 'uid': dtype('int64')}
  ab_test: ['user_id', 'revenue', 'testgroup'] | dtype: {'user_id': dtype('int64'), 'revenue': dtype('int64'), 'testgroup': dtype('O')}

      reg_ts  uid
0  911382223    1
1  932683089    2
2  947802447    3
     auth_ts  uid
0  911382223    1
1  932683089    2
2  932921206    2
   user_id  revenue testgroup
0        1        0         b
1        2        0         a
2        3        0         a


In [3]:
# 1.4 时间戳转换：Unix 秒级时间戳 → 可读日期/时间
# 保留原始 reg_ts / auth_ts 列（后续留存函数与特征工程仍需整数时间戳），同时新增日期列
reg_data['reg_date'] = pd.to_datetime(reg_data['reg_ts'], unit='s')
reg_data['reg_time'] = reg_data['reg_date'].dt.strftime('%H:%M:%S')
reg_data['date'] = reg_data['reg_date'].dt.date
reg_data['date'] = pd.to_datetime(reg_data['date'], errors='coerce')

auth_data['auth_date'] = pd.to_datetime(auth_data['auth_ts'], unit='s')
auth_data['auth_time'] = auth_data['auth_date'].dt.strftime('%H:%M:%S')
auth_data['date'] = auth_data['auth_date'].dt.date
auth_data['date'] = pd.to_datetime(auth_data['date'], errors='coerce')

# 派生月份/年份
reg_data['reg_month'] = reg_data['reg_date'].dt.to_period('M')
reg_data['reg_year'] = reg_data['reg_date'].dt.year

print('时间戳转换完成：')
print(f'  reg_data 日期范围: {reg_data["date"].min()} ~ {reg_data["date"].max()}')
print(f'  auth_data 日期范围: {auth_data["date"].min()} ~ {auth_data["date"].max()}')
print(f'  数据跨年度: {sorted(set(reg_data["reg_year"]))}')


时间戳转换完成：
  reg_data 日期范围: 1998-11-18 00:00:00 ~ 2020-09-23 00:00:00
  auth_data 日期范围: 1998-11-18 00:00:00 ~ 2020-09-23 00:00:00
  数据跨年度: [1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]


In [4]:
# 1.5 数据完整性验证：唯一用户数、UID 是否升序
reg_unique = reg_data['uid'].nunique()
auth_unique = auth_data['uid'].nunique()
ab_unique = ab_test['user_id'].nunique()

print(f'reg_data 唯一用户: {reg_unique:,}')
print(f'auth_data 唯一用户: {auth_unique:,}')
print(f'ab_test 唯一用户:  {ab_unique:,}')

# 检查 reg_data 的 uid 是否严格升序（注册表按用户创建顺序编号）
def is_sorted_ascending(column: List[int]) -> bool:
    return all(column[i] <= column[i + 1] for i in range(len(column) - 1))

print(f'reg_data uid 严格升序: {is_sorted_ascending(reg_data["uid"].values)}')
print(f'注册/登录最大最小日期: reg {reg_data["date"].min()} ~ {reg_data["date"].max()} | '
      f'auth {auth_data["date"].min()} ~ {auth_data["date"].max()}')


reg_data 唯一用户: 1,000,000
auth_data 唯一用户: 1,000,000
ab_test 唯一用户:  404,770
reg_data uid 严格升序: True
注册/登录最大最小日期: reg 1998-11-18 00:00:00 ~ 2020-09-23 00:00:00 | auth 1998-11-18 00:00:00 ~ 2020-09-23 00:00:00


## 2. 探索性数据分析 (EDA)

本阶段回答四个业务问题：注册量如何随时间增长、UID 是否存在缺口（管理员/删号）、`ab_test.user_id` 是否等价于 `uid`、以及登录/注册记录在时间轴上的缺失分布。


In [5]:
# 2.1 UID 缺失分析：是否存在被删除/保留的 UID 区间（如管理员账号）
def find_missing_numbers(column: List[int]) -> List[int]:
    '''在升序编号列中查找缺失的编号。'''
    min_val = column.min()
    max_val = column.max()
    full_range = set(range(min_val, max_val + 1))
    return sorted(full_range - set(column))

missing_reg = find_missing_numbers(reg_data['uid'])
missing_auth = find_missing_numbers(auth_data['uid'])
missing_ab = find_missing_numbers(ab_test['user_id'])

print(f'reg_data 缺失 UID 数: {len(missing_reg):,}')
print(f'auth_data 缺失 UID 数: {len(missing_auth):,}')
print(f'ab_test 缺失 user_id 数: {len(missing_ab):,}')

# 检查缺失编号中是否出现大间隔（>99），用于判断是否存在保留 UID 段
diffs = [abs(missing_reg[i] - missing_reg[i - 1]) for i in range(1, len(missing_reg))]
large_gaps = [d for d in diffs if d > 99]
print(f'reg_data 缺失编号中 >99 的大间隔个数: {len(large_gaps)}，最大间隔: {max(large_gaps) if large_gaps else "无"}')
print('结论：无法确认管理员 UID；缺失编号更可能是删号/不活跃导致的玩家账号清除。')


reg_data 缺失 UID 数: 110,622
auth_data 缺失 UID 数: 110,622
ab_test 缺失 user_id 数: 0
reg_data 缺失编号中 >99 的大间隔个数: 4，最大间隔: 124
结论：无法确认管理员 UID；缺失编号更可能是删号/不活跃导致的玩家账号清除。


In [6]:
# 2.2 ab_test.user_id 是否等于 uid 的关联校验
merged_ab_reg = pd.merge(ab_test, reg_data, left_on='user_id', right_on='uid', how='inner')
print(f'ab_test 总记录: {len(ab_test):,}')
print(f'ab_test ∩ reg_data (inner join): {len(merged_ab_reg):,}')
print(f'未匹配记录: {len(ab_test) - len(merged_ab_reg):,}')

ab_uid_set = set(ab_test['user_id'])
reg_uid_set = set(reg_data['uid'])
unmatched = ab_uid_set - reg_uid_set
print(f'ab_test 中不在 reg_data 的 user_id 数量: {len(unmatched):,}')
print('结论：ab_test.user_id 与 reg_data.uid 并不完全一致（存在 40,215 个差异），A/B 表是独立编号的实验人群。')

print('\nA/B 测试分组规模：')
print(ab_test.groupby('testgroup').size().to_string())


ab_test 总记录: 404,770
ab_test ∩ reg_data (inner join): 364,555
未匹配记录: 40,215
ab_test 中不在 reg_data 的 user_id 数量: 40,215
结论：ab_test.user_id 与 reg_data.uid 并不完全一致（存在 40,215 个差异），A/B 表是独立编号的实验人群。

A/B 测试分组规模：
testgroup
a    202103
b    202667


In [7]:
# 2.3 新玩家注册趋势：月度（2010 年起）与年度（含同比增速）
reg_2010 = reg_data[reg_data['date'] >= '2010-01-01']
monthly_growth = reg_2010.groupby('reg_month')['uid'].nunique()

yearly = reg_data.groupby('reg_year')['uid'].nunique()
growth_pct = yearly.pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

ax1 = axes[0]
ax1.plot(monthly_growth.index.astype(str), monthly_growth.values, marker='o', color='#2196F3')
ax1.set_title('2010 年以来月度新玩家注册量', fontweight='bold')
ax1.set_xlabel('月份')
ax1.set_ylabel('新玩家数量')
ax1.tick_params(axis='x', rotation=90)
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
colors = plt.cm.viridis(np.linspace(0.4, 1, len(yearly)))
bars = ax2.bar(yearly.index.astype(str), yearly.values, color=colors)
ax2.plot(yearly.index.astype(str), yearly.values, marker='o', color='red', linewidth=2)
ax2.set_title('年度新玩家注册量与同比增速', fontweight='bold')
ax2.set_xlabel('年份')
ax2.set_ylabel('新玩家数量')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True, alpha=0.3, axis='y')
for i, bar in enumerate(bars):
    if i > 0:
        pct = growth_pct.iloc[i]
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
                 f'{pct:.1f}%', ha='center', fontsize=8, color='black')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '02_registration_trend.png'), dpi=150, bbox_inches='tight')
plt.show()

peak_year = yearly.idxmax()
print(f'注册峰值年份: {peak_year}（{yearly.max():,} 人）')
print(f'2018 年同比增速: {growth_pct.loc[2018]:.1f}% | 2020 年同比增速: {growth_pct.loc[2020]:.1f}%')
print('结论：2016 年起注册呈爆发式增长，2018 年增速达 82.2%，2020 年仍有 21.9% 增长。')


注册峰值年份: 2020（354,963 人）
2018 年同比增速: 82.2% | 2020 年同比增速: 21.9%
结论：2016 年起注册呈爆发式增长，2018 年增速达 82.2%，2020 年仍有 21.9% 增长。


In [8]:
# 2.4 缺失日期热力图：按年-月统计注册/登录记录的缺失天数
# 用于识别数据采集的中断与游戏冷启动期（1998 年起始，早期数据稀疏）
def missed_dates(df_data, date_col, start_date='1998-11-18', end_date='2020-09-23'):
    '''统计指定日期范围内，每个年-月缺失的天数，并返回最后一个缺失日期。'''
    df_data = df_data.copy()
    df_data[date_col] = pd.to_datetime(df_data[date_col])
    present_dates = set(df_data[date_col].dt.date)
    full_range = pd.date_range(start=start_date, end=end_date, freq='D')
    missing = [d for d in full_range if d.date() not in present_dates]
    missing_df = pd.DataFrame({'date': missing})
    missing_df['year'] = missing_df['date'].dt.year
    missing_df['month'] = missing_df['date'].dt.month
    missing_count = missing_df.groupby(['year', 'month']).size().unstack(fill_value=0)
    last_missing = missing_df['date'].max() if len(missing_df) else None
    return missing_count, last_missing

missing_auth, last_missing_auth = missed_dates(auth_data, 'date')
missing_reg, last_missing_reg = missed_dates(reg_data, 'date')

fig, axes = plt.subplots(2, 1, figsize=(14, 14))
sns.heatmap(missing_auth, cmap='YlGnBu', linewidths=.5, annot=True, fmt='d', ax=axes[0])
axes[0].set_title('活动数据缺失天数（按年-月）', fontweight='bold')
axes[0].set_xlabel('月份')
axes[0].set_ylabel('年份')
sns.heatmap(missing_reg, cmap='YlGnBu', linewidths=.5, annot=True, fmt='d', ax=axes[1])
axes[1].set_title('注册数据缺失天数（按年-月）', fontweight='bold')
axes[1].set_xlabel('月份')
axes[1].set_ylabel('年份')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '02_missing_dates_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'活动数据最后缺失日期: {last_missing_auth.date()}')
print(f'注册数据最后缺失日期: {last_missing_reg.date()}')
print('结论：1999-2006 年为数据稀疏期，2007 年后缺失天数显著减少，游戏进入稳定运营期。')


活动数据最后缺失日期: 2007-06-26
注册数据最后缺失日期: 2008-05-14
结论：1999-2006 年为数据稀疏期，2007 年后缺失天数显著减少，游戏进入稳定运营期。


In [9]:
# 2.5 季节性洞察：二月「注册下降」与 2020 年 9 月「数据截断」
# (a) 二月：用「日均注册量」消除月份天数差异后，判断二月是否真实下降
days_in_month = {1: 31, 2: 28.25, 3: 31, 4: 30, 5: 31, 6: 30,
                 7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31}

monthly_reg = reg_data.groupby(['reg_year', reg_data['date'].dt.month]).size().reset_index(name='registrations')
monthly_reg.columns = ['year', 'month', 'registrations']
monthly_reg['days_in_month'] = monthly_reg['month'].apply(lambda x: days_in_month[x])
monthly_reg['avg_per_day'] = monthly_reg['registrations'] / monthly_reg['days_in_month']

feb = monthly_reg[monthly_reg['month'] == 2]
mean_feb = feb['avg_per_day'].mean()
mean_other = monthly_reg[monthly_reg['month'] != 2]['avg_per_day'].mean()
print(f'二月日均注册量: {mean_feb:.2f}')
print(f'其他月份日均注册量: {mean_other:.2f}')
print('结论：' + ('二月出现真实趋势下降。' if mean_feb < mean_other else '二月未出现真实趋势下降。'))

# (b) 2020 年 9 月：按日注册量，观察尾部截断
sept = reg_data[(reg_data['date'] >= '2020-09-01') & (reg_data['date'] <= '2020-09-30')]
regs_per_day = sept['date'].dt.day.value_counts().sort_index()
plt.figure(figsize=(10, 5))
plt.bar(regs_per_day.index, regs_per_day.values, color='#2196F3')
plt.xlabel('2020 年 9 月的日期')
plt.ylabel('注册数量')
plt.title('2020 年 9 月每日注册量（尾部为数据截断）', fontweight='bold')
plt.xticks(range(1, 31))
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '02_sept2020.png'), dpi=150, bbox_inches='tight')
plt.show()
print('结论：数据集最大注册日期为 2020-09-23，9 月 23 日后的下降是数据截断而非业务下滑。')


二月日均注册量: 134.41
其他月份日均注册量: 140.25
结论：二月出现真实趋势下降。


结论：数据集最大注册日期为 2020-09-23，9 月 23 日后的下降是数据截断而非业务下滑。


## 3. 指标体系与留存分析（任务 1）

留存率是游戏运营最重要的指标之一。本章实现两类留存口径：

- **每日留存率**：`第 N 天留存率 = 注册后第 N 天登录过的玩家数 / 总注册玩家数 × 100%`，用 Unix 整数运算高效实现；
- **月度群组留存率 (Cohort)**：按注册月份分组，绘制「注册月份 × 后续月份」的留存热力图，观察不同群组的留存衰减曲线。


In [10]:
# 3.1 每日留存率函数（高效向量化实现，支持抽样间隔与天数上限）
def calculate_daily_retention(reg_data_raw, auth_data_raw, sample_interval=10, max_days=None):
    '''计算每日留存率。

    参数:
        reg_data_raw (DataFrame): 含 reg_ts, uid 的注册表
        auth_data_raw (DataFrame): 含 auth_ts, uid 的登录表
        sample_interval (int): 抽样间隔（天），0 表示不抽样
        max_days (int): 天数上限，None 表示不限制

    返回:
        DataFrame: 含 days_since_reg, login_count(去重用户), login_percent
    '''
    merged = pd.merge(reg_data_raw, auth_data_raw, on='uid', how='left')
    merged['days_since_reg'] = (merged['auth_ts'] - merged['reg_ts']) // (60 * 60 * 24)

    login_count = (
        merged.groupby('days_since_reg')['uid']
        .nunique()
        .reset_index(name='login_count')
        .sort_values('days_since_reg')
    )
    total_users = merged['uid'].nunique()
    login_count['login_percent'] = (login_count['login_count'] / total_users) * 100

    if sample_interval and sample_interval > 0:
        results = login_count[login_count['days_since_reg'] % sample_interval == 0].reset_index(drop=True)
    else:
        results = login_count.reset_index(drop=True)
    if max_days is not None:
        results = results[results['days_since_reg'] <= max_days]
    return results

retention_10 = calculate_daily_retention(reg_data, auth_data, sample_interval=10)
retention_30 = calculate_daily_retention(reg_data, auth_data, sample_interval=30)

print('每日留存率（每 10 天抽样，前 20 行）：')
print(retention_10[['days_since_reg', 'login_percent']].head(20).to_string(index=False))
print(f'...（共 {len(retention_10)} 个抽样点）')


每日留存率（每 10 天抽样，前 20 行）：
 days_since_reg  login_percent
              0       100.0000
             10         5.1070
             20         3.7987
             30         2.6259
             40         1.5905
             50         1.1790
             60         1.1293
             70         1.1338
             80         1.1157
             90         1.0839
            100         1.0716
            110         1.0460
            120         1.0336
            130         1.0181
            140         0.9946
            150         1.0001
            160         0.9508
            170         0.9397
            180         0.9302
            190         0.9250
...（共 595 个抽样点）


In [11]:
# 3.2 留存率可视化：全周期（每 30 天）与前 90 天（每 10 天）
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

ax1 = axes[0]
ret_30 = retention_30[retention_30['days_since_reg'] <= 365]
ax1.plot(ret_30['days_since_reg'], ret_30['login_percent'], marker='o', color='#E91E63', linewidth=2, markersize=4)
ax1.set_title('年度留存率趋势（每 30 天）', fontweight='bold')
ax1.set_xlabel('注册后天数')
ax1.set_ylabel('留存率 (%)')
ax1.axhline(y=1, color='red', linestyle='--', alpha=0.5, label='1% 基准线')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
ret_10_short = retention_10[retention_10['days_since_reg'] <= 90]
ax2.plot(ret_10_short['days_since_reg'], ret_10_short['login_percent'], marker='o', color='#2196F3', linewidth=2, markersize=6)
ax2.set_title('前 90 天留存率（每 10 天）', fontweight='bold')
ax2.set_xlabel('注册后天数')
ax2.set_ylabel('留存率 (%)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03_daily_retention.png'), dpi=150, bbox_inches='tight')
plt.show()

print('关键节点留存率：')
for day in [1, 3, 5, 7, 14, 30, 60, 90]:
    v = retention_10[retention_10['days_since_reg'] == day]
    if len(v) > 0:
        print(f'  第 {day:>3d} 天留存率: {v["login_percent"].values[0]:.2f}%')


关键节点留存率：
  第  30 天留存率: 2.63%
  第  60 天留存率: 1.13%
  第  90 天留存率: 1.08%


In [12]:
# 3.3 月度群组留存 (Cohort) 热力图
def calculate_cohort_retention(reg_data, auth_data, start_date='2015-01', end_date='2020-12'):
    '''计算按月群组留存率矩阵（行=注册月份，列=后续周期编号）。'''
    reg_copy = reg_data.copy()
    auth_copy = auth_data.copy()
    reg_copy['reg_month'] = reg_copy['date'].dt.to_period('M')

    data = pd.merge(auth_copy, reg_copy[['uid', 'reg_month']], on='uid', how='left')
    data['active_month'] = data['date'].dt.to_period('M')
    data = data[(data['reg_month'] >= start_date) & (data['reg_month'] <= end_date)]

    cohorts = data.groupby(['reg_month', 'active_month']).agg({'uid': 'nunique'}).reset_index()
    cohorts['period_number'] = (cohorts['active_month'] - cohorts['reg_month']).apply(lambda x: x.n)
    cohort_pivot = cohorts.pivot_table(index='reg_month', columns='period_number', values='uid')
    cohort_size = cohort_pivot.iloc[:, 0]
    retention_matrix = cohort_pivot.divide(cohort_size, axis=0)
    return retention_matrix

retention_matrix = calculate_cohort_retention(reg_data, auth_data)

fig, ax = plt.subplots(figsize=(20, 10))
sns.heatmap(retention_matrix.iloc[:, :12], annot=True, fmt='.0%', cmap='Blues', ax=ax,
            cbar_kws={'format': '%.0f%%'})
ax.set_title('月度群组留存率热力图（前 12 个周期）', fontsize=16, fontweight='bold')
ax.set_xlabel('注册后月份')
ax.set_ylabel('群组注册月份')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03_cohort_retention.png'), dpi=150, bbox_inches='tight')
plt.show()

# 首月留存（period=1，即注册次月）在 2015-2020 群组中的真实区间
p1 = retention_matrix[1].dropna()
print(f'首月留存率（period=1, 2015-2020 群组）：')
print(f'  均值 {p1.mean()*100:.2f}% | 中位数 {p1.median()*100:.2f}% | 区间 {p1.min()*100:.2f}%~{p1.max()*100:.2f}%')


首月留存率（period=1, 2015-2020 群组）：
  均值 17.56% | 中位数 17.52% | 区间 15.43%~18.87%


In [13]:
# 3.4 玩家生命周期分布（首次注册到最后登录的天数，使用完整时间戳保证精确口径）
player_last_login = auth_data.groupby('uid')['auth_date'].max().reset_index()
player_last_login.columns = ['uid', 'last_login_date']
lifecycle = pd.merge(
    reg_data[['uid', 'reg_date']],
    player_last_login, on='uid', how='inner'
)
lifecycle['lifecycle_days'] = (lifecycle['last_login_date'] - lifecycle['reg_date']).dt.days

bins = [0, 1, 7, 30, 90, 365, float('inf')]
labels = ['1天', '2-7天', '8-30天', '31-90天', '91-365天', '365天+']
lifecycle['lifecycle_tier'] = pd.cut(lifecycle['lifecycle_days'], bins=bins, labels=labels)
tier_counts = lifecycle['lifecycle_tier'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_life = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(tier_counts)))
axes[0].bar(range(len(tier_counts)), tier_counts.values, color=colors_life)
axes[0].set_xticks(range(len(tier_counts)))
axes[0].set_xticklabels(tier_counts.index, rotation=30)
axes[0].set_title('玩家生命周期分布', fontweight='bold')
axes[0].set_xlabel('生命周期长度')
axes[0].set_ylabel('玩家数量')
for i, v in enumerate(tier_counts.values):
    axes[0].text(i, v + 3000, f'{v:,}\n({v/len(lifecycle)*100:.1f}%)', ha='center', fontsize=9)

axes[1].hist(lifecycle['lifecycle_days'].clip(upper=365), bins=50, color='#673AB7', edgecolor='white', alpha=0.7)
axes[1].axvline(lifecycle['lifecycle_days'].median(), color='red', linestyle='--',
                linewidth=2, label=f'中位数: {lifecycle["lifecycle_days"].median():.0f}天')
axes[1].set_title('生命周期（天）分布（截断至 365 天）', fontweight='bold')
axes[1].set_xlabel('生命周期 (天)')
axes[1].set_ylabel('玩家数量')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03_lifecycle.png'), dpi=150, bbox_inches='tight')
plt.show()

# 关键流失数字（后续简历/报告引用的真实复算值）
early_churn_rate = (lifecycle['lifecycle_days'] <= 7).mean() * 100
one_day_rate = (lifecycle['lifecycle_days'] <= 1).mean() * 100
print(f'早期流失率（生命周期 ≤7 天）: {early_churn_rate:.2f}%  ({(lifecycle["lifecycle_days"]<=7).sum():,} / {len(lifecycle):,})')
print(f'仅 1 天流失率: {one_day_rate:.2f}%')
print(f'生命周期中位数: {lifecycle["lifecycle_days"].median():.0f} 天 | 均值: {lifecycle["lifecycle_days"].mean():.1f} 天')


早期流失率（生命周期 ≤7 天）: 79.42%  (794,152 / 1,000,000)
仅 1 天流失率: 76.50%
生命周期中位数: 0 天 | 均值: 34.2 天


## 4. A/B 测试与增长量化（任务 2）

背景：两组玩家被提供不同促销方案。对照组 (a) 202,103 名用户中 1,928 名付费；测试组 (b) 202,667 名用户中 1,805 名付费。

本阶段用 **ARPU / CR / ARPPU** 三维指标 + 统计显著性检验 + **Cohen's d 效应量评分模型**，综合判定最优促销方案。


In [14]:
# 4.1 基础指标：ARPU / CR / ARPPU
control = ab_test[ab_test['testgroup'] == 'a']
test = ab_test[ab_test['testgroup'] == 'b']

arpu_control, arpu_test = control['revenue'].mean(), test['revenue'].mean()
cr_control = (control['revenue'] > 0).mean() * 100
cr_test = (test['revenue'] > 0).mean() * 100
arppu_control = control[control['revenue'] > 0]['revenue'].mean()
arppu_test = test[test['revenue'] > 0]['revenue'].mean()

metrics_df = pd.DataFrame({
    '指标': ['ARPU ($)', '转化率 CR (%)', 'ARPPU ($)', '用户数', '付费用户数'],
    '对照组 (a)': [f'{arpu_control:.2f}', f'{cr_control:.4f}%', f'{arppu_control:.2f}',
                 f'{len(control):,}', f'{(control["revenue"]>0).sum():,}'],
    '测试组 (b)': [f'{arpu_test:.2f}', f'{cr_test:.4f}%', f'{arppu_test:.2f}',
                 f'{len(test):,}', f'{(test["revenue"]>0).sum():,}'],
    '差异': [f'{arpu_test - arpu_control:+.2f}', f'{cr_test - cr_control:+.4f}%',
            f'{arppu_test - arppu_control:+.2f}', '', '']
})
print(metrics_df.to_string(index=False))
print(f'\nARPPU 相对提升: {(arppu_test - arppu_control) / arppu_control * 100:.2f}%')
print(f'CR 差异: {cr_test - cr_control:+.4f} pp')


        指标 对照组 (a) 测试组 (b)       差异
  ARPU ($)   25.41   26.75    +1.34
转化率 CR (%) 0.9540% 0.8906% -0.0633%
 ARPPU ($) 2664.00 3003.66  +339.66
       用户数 202,103 202,667         
     付费用户数   1,928   1,805         

ARPPU 相对提升: 12.75%
CR 差异: -0.0633 pp


In [15]:
# 4.2 统计检验：正态性、方差齐性、ARPU/CR/ARPPU 差异、置信区间
print('=' * 70)
print('统计显著性检验')
print('=' * 70)

# (1) 正态性检验 (D'Agostino-Pearson)
stat_a, p_a = stats.normaltest(control['revenue'])
stat_b, p_b = stats.normaltest(test['revenue'])
print(f'\n正态性检验 (D\'Agostino-Pearson):')
print(f'  对照组: stat={stat_a:.2f}, p={p_a:.6f} -> {"正态" if p_a > 0.05 else "非正态"}')
print(f'  测试组: stat={stat_b:.2f}, p={p_b:.6f} -> {"正态" if p_b > 0.05 else "非正态"}')

# (2) 方差齐性 (Levene)
lv_stat, lv_p = stats.levene(control['revenue'], test['revenue'])
print(f'\n方差齐性 (Levene): stat={lv_stat:.4f}, p={lv_p:.4f} -> {"方差齐" if lv_p > 0.05 else "方差不齐"}')

# (3) ARPU 差异 (Mann-Whitney U，非参数)
mw_stat, mw_p = stats.mannwhitneyu(control['revenue'], test['revenue'], alternative='two-sided')
print(f'\nARPU 差异 (Mann-Whitney U): U={mw_stat:,.0f}, p={mw_p:.6f} -> {"显著" if mw_p < 0.05 else "不显著"}')

# (4) 转化率差异 (卡方检验)
contingency = np.array([
    [(control['revenue'] > 0).sum(), (control['revenue'] == 0).sum()],
    [(test['revenue'] > 0).sum(), (test['revenue'] == 0).sum()]
])
chi2, chi2_p, dof, expected = chi2_contingency(contingency)
print(f'\n转化率差异 (卡方): chi2={chi2:.4f}, p={chi2_p:.6f} -> {"显著" if chi2_p < 0.05 else "不显著"}')

# (5) ARPPU 差异 (Mann-Whitney U)
payers_a = control[control['revenue'] > 0]['revenue']
payers_b = test[test['revenue'] > 0]['revenue']
arppu_u, arppu_p = stats.mannwhitneyu(payers_a, payers_b, alternative='two-sided')
print(f'\nARPPU 差异 (Mann-Whitney U): U={arppu_u:,.0f}, p={arppu_p:.6f} -> {"显著" if arppu_p < 0.05 else "不显著"}')

# (6) ARPU 95% 置信区间
def calc_ci(data, confidence=0.95):
    mean = np.mean(data)
    std = np.std(data, ddof=1)
    n = len(data)
    ci = stats.t.interval(confidence, n - 1, loc=mean, scale=std / np.sqrt(n))
    return mean, ci

mean_a, ci_a = calc_ci(control['revenue'])
mean_b, ci_b = calc_ci(test['revenue'])
print(f'\nARPU 95% 置信区间:')
print(f'  对照组: {mean_a:.2f} [{ci_a[0]:.2f}, {ci_a[1]:.2f}]')
print(f'  测试组: {mean_b:.2f} [{ci_b[0]:.2f}, {ci_b[1]:.2f}]')


统计显著性检验

正态性检验 (D'Agostino-Pearson):
  对照组: stat=585152.73, p=0.000000 -> 非正态
  测试组: stat=326622.40, p=0.000000 -> 非正态

方差齐性 (Levene): stat=0.3896, p=0.5325 -> 方差齐

ARPU 差异 (Mann-Whitney U): U=20,491,259,376, p=0.062697 -> 不显著

转化率差异 (卡方): chi2=4.3747, p=0.036476 -> 显著

ARPPU 差异 (Mann-Whitney U): U=222,015, p=0.000000 -> 显著

ARPU 95% 置信区间:
  对照组: 25.41 [21.40, 29.43]
  测试组: 26.75 [25.50, 28.00]


In [16]:
# 4.3 可视化：Q-Q 图、箱线图、收入分布、指标对比
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, (group, name, color) in enumerate([
    (control['revenue'], '对照组 (a)', '#2196F3'),
    (test['revenue'], '测试组 (b)', '#FF9800')
]):
    stats.probplot(group, dist='norm', plot=axes[0, i])
    axes[0, i].set_title(f'{name} Q-Q 图', fontweight='bold')

bp = axes[0, 2].boxplot([control['revenue'], test['revenue']], labels=['对照组 (a)', '测试组 (b)'], patch_artist=True)
bp['boxes'][0].set_facecolor('#2196F3')
bp['boxes'][1].set_facecolor('#FF9800')
axes[0, 2].set_title('收入分布箱线图', fontweight='bold')
axes[0, 2].set_ylabel('收入 ($)')

for i, (group, name, color) in enumerate([
    (payers_a, '对照组 (a)', '#2196F3'),
    (payers_b, '测试组 (b)', '#FF9800')
]):
    axes[1, i].hist(group, bins=30, color=color, alpha=0.7, edgecolor='white')
    axes[1, i].axvline(group.mean(), color='red', linestyle='--', linewidth=2, label=f'均值: {group.mean():.0f}')
    axes[1, i].set_title(f'{name} 付费用户收入分布', fontweight='bold')
    axes[1, i].set_xlabel('收入 ($)')
    axes[1, i].set_ylabel('频次')
    axes[1, i].legend()

metrics_names = ['ARPU\n($)', '转化率\n(%)', 'ARPPU\n(÷100)']
metrics_a = [arpu_control, cr_control, arppu_control / 100]
metrics_b = [arpu_test, cr_test, arppu_test / 100]
x = np.arange(len(metrics_names))
width = 0.35
axes[1, 2].bar(x - width/2, metrics_a, width, label='对照组 (a)', color='#2196F3')
axes[1, 2].bar(x + width/2, metrics_b, width, label='测试组 (b)', color='#FF9800')
axes[1, 2].set_xticks(x)
axes[1, 2].set_xticklabels(metrics_names)
axes[1, 2].set_title('关键指标对比', fontweight='bold')
axes[1, 2].legend()
axes[1, 2].set_ylabel('值')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '04_ab_test_visualization.png'), dpi=150, bbox_inches='tight')
plt.show()


In [17]:
# 4.4 Cohen's d 效应量评分模型：将 A/B 组在多维指标上的差异映射为 0-100 分
# 分差 = min(50, |d| × 25)，基准 50；|d|=0.2(小)→55vs45 |d|=0.5(中)→62.5vs37.5 |d|≥2.0→100vs0
def _cohens_d_score(vals_a, vals_b, higher_is_better=True):
    '''Cohen's d 效应量 → 分数映射。'''
    mean_a, mean_b = np.mean(vals_a), np.mean(vals_b)
    var_a, var_b = np.var(vals_a, ddof=1), np.var(vals_b, ddof=1)
    n_a, n_b = len(vals_a), len(vals_b)
    pooled_std = np.sqrt(((n_a - 1) * var_a + (n_b - 1) * var_b) / (n_a + n_b - 2))
    d = (mean_a - mean_b) / pooled_std if pooled_std >= 1e-15 else 0.0
    score_diff = min(50.0, abs(d) * 25.0)
    a_better = (d > 0) if higher_is_better else (d < 0)
    if a_better:
        return 50.0 + score_diff, 50.0 - score_diff, float(d)
    elif abs(d) < 1e-15:
        return 50.0, 50.0, float(d)
    else:
        return 50.0 - score_diff, 50.0 + score_diff, float(d)


def score_ab_groups(ab_df, weights=None, verbose=True):
    '''基于 4 个指标（付费转化率/ARPU/ARPPU/MAD）为 A/B 组综合打分。'''
    if weights is None:
        weights = {'percent_paying': 0.4, 'arpu': 0.3, 'arppu': 0.2, 'mad': 0.1}
    groups = sorted(ab_df['testgroup'].unique())
    g_a, g_b = groups[0], groups[1]
    rev_a = ab_df[ab_df['testgroup'] == g_a]['revenue'].values
    rev_b = ab_df[ab_df['testgroup'] == g_b]['revenue'].values
    payers_a, payers_b = rev_a[rev_a > 0], rev_b[rev_b > 0]

    pct_a, pct_b = (rev_a > 0).mean() * 100, (rev_b > 0).mean() * 100
    arpu_a, arpu_b = rev_a.mean(), rev_b.mean()
    arppu_a = payers_a.mean() if len(payers_a) > 0 else 0.0
    arppu_b = payers_b.mean() if len(payers_b) > 0 else 0.0
    mad_a = np.abs(rev_a - arpu_a).mean()
    mad_b = np.abs(rev_b - arpu_b).mean()

    s_pct_a, s_pct_b, d_pct = _cohens_d_score((rev_a > 0).astype(float), (rev_b > 0).astype(float), True)
    s_arpu_a, s_arpu_b, d_arpu = _cohens_d_score(rev_a, rev_b, True)
    if len(payers_a) >= 2 and len(payers_b) >= 2:
        s_arppu_a, s_arppu_b, d_arppu = _cohens_d_score(payers_a, payers_b, True)
    else:
        s_arppu_a, s_arppu_b, d_arppu = 50.0, 50.0, np.nan
    s_mad_a, s_mad_b, d_mad = _cohens_d_score(np.abs(rev_a - arpu_a), np.abs(rev_b - arpu_b), False)

    composite_a = (s_pct_a * weights['percent_paying'] + s_arpu_a * weights['arpu'] +
                   s_arppu_a * weights['arppu'] + s_mad_a * weights['mad'])
    composite_b = (s_pct_b * weights['percent_paying'] + s_arpu_b * weights['arpu'] +
                   s_arppu_b * weights['arppu'] + s_mad_b * weights['mad'])

    def _d_label(d_val):
        ad = abs(d_val)
        return '可忽略' if ad < 0.2 else '小差异' if ad < 0.5 else '中等差异' if ad < 0.8 else '大差异'

    if verbose:
        print('=' * 70)
        print('A/B 测试组综合评分报告（Cohen\'s d 效应量驱动）')
        print('=' * 70)
        print(f"  {'指标':<14s} {'组 ' + g_a:>10s} {'组 ' + g_b:>10s} {'差异':>10s} {'d':>9s} {'解读'}")
        print(f"  {'付费转化率(%)':<14s} {pct_a:>10.4f} {pct_b:>10.4f} {pct_a-pct_b:>+10.4f} {d_pct:>9.4f} {_d_label(d_pct)}")
        print(f"  {'ARPU($)':<14s} {arpu_a:>10.4f} {arpu_b:>10.4f} {arpu_a-arpu_b:>+10.4f} {d_arpu:>9.4f} {_d_label(d_arpu)}")
        print(f"  {'ARPPU($)':<14s} {arppu_a:>10.4f} {arppu_b:>10.4f} {arppu_a-arppu_b:>+10.4f} {d_arppu:>9.4f} {_d_label(d_arppu)}")
        print(f"  {'MAD($)':<14s} {mad_a:>10.4f} {mad_b:>10.4f} {mad_a-mad_b:>+10.4f} {d_mad:>9.4f} {_d_label(d_mad)}")
        print(f"\n综合得分（权重={weights}）: 组 '{g_a}': {composite_a:.2f} | 组 '{g_b}': {composite_b:.2f}")
        gap = abs(composite_a - composite_b)
        winner = g_a if composite_a > composite_b else g_b
        print(f'  胜出组: {winner}（领先 {gap:.2f} 分）')

    return {'composite': {g_a: composite_a, g_b: composite_b},
            'metrics': {'arpu': {g_a: arpu_a, g_b: arpu_b}, 'cr': {g_a: pct_a, g_b: pct_b},
                        'arppu': {g_a: arppu_a, g_b: arppu_b}}}

ab_scores = score_ab_groups(ab_test)


A/B 测试组综合评分报告（Cohen's d 效应量驱动）
  指标                    组 a        组 b         差异         d 解读
  付费转化率(%)           0.9540     0.8906    +0.0633    0.0066 可忽略
  ARPU($)           25.4137    26.7513    -1.3376   -0.0020 可忽略
  ARPPU($)        2663.9984  3003.6582  -339.6597   -0.0521 可忽略
  MAD($)            50.3426    53.0261    -2.6835   -0.0039 可忽略

综合得分（权重={'percent_paying': 0.4, 'arpu': 0.3, 'arppu': 0.2, 'mad': 0.1}）: 组 'a': 49.80 | 组 'b': 50.20
  胜出组: b（领先 0.40 分）


### 4.5 A/B 测试结论

| 指标 | 对照组 (a) | 测试组 (b) | 差异显著性 | 胜出 |
|------|-----------|-----------|-----------|------|
| ARPU | $25.41 | $26.75 | 不显著 (p=0.0627) | — |
| 转化率 CR | 0.954% | 0.891% | 显著 (p=0.0365) | 对照组 |
| ARPPU | $2,664 | $3,004 | 显著 (p<0.001) | **测试组** |

**决策：** 测试组 ARPPU 显著提升 12.75%（每付费用户价值更高），而 CR 仅微降 0.06pp。结合 Cohen's d 评分，**建议实施测试组 (b) 促销方案**——以极小的付费渗透率代价，换取显著的单付费用户价值提升。


## 5. 玩家画像与行为聚类

本章从「登录行为」与「消费行为」两个维度刻画玩家画像：

1. **登录间隔**：计算每位玩家相邻登录的时间间隔，识别参与度层级与风险玩家；
2. **消费-留存关系**：将收入数据与行为特征打通，发现「消费与留存呈倒 U 型」的核心洞察；
3. **无监督聚类**：用 K-Means / DBSCAN / GMM 三种算法自动发现隐藏的玩家行为原型。


In [18]:
# 5.1 登录间隔分析：按玩家排序后计算相邻登录的天数差
auth_sorted = auth_data.sort_values(['uid', 'auth_date']).copy()
auth_sorted['days_since_last_login'] = auth_sorted.groupby('uid')['auth_date'].diff().dt.days

print('登录间隔（days_since_last_login）描述统计：')
print(auth_sorted['days_since_last_login'].describe().to_string())

long_gap = auth_sorted[auth_sorted['days_since_last_login'] > 30]
print(f'\n长间隔记录（>30 天未登录）: {len(long_gap):,} 条，占比 {len(long_gap)/len(auth_sorted)*100:.2f}%')


登录间隔（days_since_last_login）描述统计：


count    8.601013e+06
mean     3.495116e+00
std      1.707709e+00
min      1.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      5.000000e+00
max      7.000000e+00

长间隔记录（>30 天未登录）: 0 条，占比 0.00%


In [19]:
# 5.2 玩家参与度画像：平均登录间隔、稳定性、生命周期、风险评分
# 平均登录间隔（玩家级）
player_gap_summary = auth_sorted.groupby('uid')['days_since_last_login'].mean().reset_index()
player_gap_summary.columns = ['uid', 'days_since_last_login']

# 登录间隔标准差（行为稳定性）
player_stability = auth_sorted.groupby('uid')['days_since_last_login'].std().reset_index()
player_stability.columns = ['uid', 'gap_std']
player_gap_summary = player_gap_summary.merge(player_stability, on='uid', how='left')

# 生命周期（注册到最后登录天数，完整时间戳口径）
player_last_login = auth_data.groupby('uid')['auth_date'].max().reset_index()
player_last_login.columns = ['uid', 'last_login_date']
player_lifecycle = reg_data[['uid', 'reg_date']].merge(
    player_last_login, on='uid', how='inner')
player_lifecycle['lifecycle_days'] = (player_lifecycle['last_login_date'] - player_lifecycle['reg_date']).dt.days
player_gap_summary = player_gap_summary.merge(player_lifecycle[['uid', 'lifecycle_days']], on='uid', how='left')

player_gap_summary = player_gap_summary.dropna()

# 参与度分层
def segment_player(gap):
    if gap <= 2:
        return '高参与度 (≤2天)'
    elif gap <= 4:
        return '中等参与度 (2-4天)'
    else:
        return '低参与度 (>4天)'

player_gap_summary['player_segment'] = player_gap_summary['days_since_last_login'].apply(segment_player)

# 早期流失标签（生命周期 ≤ 7 天）
player_gap_summary['early_churn'] = (player_gap_summary['lifecycle_days'] <= 7)

# 启发式行为风险评分：登录间隔越久、越不稳定、生命周期越短 → 风险越高
player_gap_summary['risk_score'] = (
    player_gap_summary['days_since_last_login'] * 0.4 +
    player_gap_summary['gap_std'] * 0.4 -
    player_gap_summary['lifecycle_days'] * 0.0005
)

# 风险玩家筛选（长间隔 + 高不稳定）
at_risk_players = player_gap_summary[
    (player_gap_summary['days_since_last_login'] > 4.5) &
    (player_gap_summary['gap_std'] > 1.5)
]

print('玩家画像汇总：')
print(f'  玩家总数: {len(player_gap_summary):,}')
print(f'  早期流失率: {player_gap_summary["early_churn"].mean()*100:.2f}%')
print(f'  风险玩家（长间隔+高不稳定）: {len(at_risk_players):,} 人')
print('\n参与度分层分布：')
print(player_gap_summary['player_segment'].value_counts().to_string())


玩家画像汇总：
  玩家总数: 217,976
  早期流失率: 5.56%
  风险玩家（长间隔+高不稳定）: 3,805 人

参与度分层分布：
player_segment
中等参与度 (2-4天)    173730
低参与度 (>4天)       36456
高参与度 (≤2天)        7790


In [20]:
# 5.3 参与度分层可视化：平均间隔分布 + 分层占比
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(player_gap_summary['days_since_last_login'], bins=30, color='#2196F3', edgecolor='white', alpha=0.7)
axes[0].axvline(player_gap_summary['days_since_last_login'].median(), color='red', linestyle='--',
                linewidth=2, label=f'中位数: {player_gap_summary["days_since_last_login"].median():.2f}天')
axes[0].set_title('玩家平均登录间隔分布', fontweight='bold')
axes[0].set_xlabel('平均间隔 (天)')
axes[0].set_ylabel('玩家数量')
axes[0].legend()

seg_counts = player_gap_summary['player_segment'].value_counts()
axes[1].pie(seg_counts.values, labels=seg_counts.index, colors=['#4CAF50', '#FF9800', '#F44336'],
            autopct='%1.1f%%', explode=(0.02, 0.02, 0.02))
axes[1].set_title('玩家参与度分层', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '05_engagement_segments.png'), dpi=150, bbox_inches='tight')
plt.show()


In [21]:
# 5.4 消费-留存关系（倒 U 型）：将 A/B 收入数据与行为特征打通
reg_uid_set = set(reg_data['uid'])
matched_ab = ab_test[ab_test['user_id'].isin(reg_uid_set)]

retention_raw = pd.merge(reg_data[['uid', 'reg_ts']], auth_data[['uid', 'auth_ts']], on='uid', how='left')
combined = pd.merge(retention_raw, matched_ab[['user_id', 'revenue', 'testgroup']],
                    left_on='uid', right_on='user_id', how='left')
combined['revenue'] = combined['revenue'].fillna(0)
combined['testgroup'] = combined['testgroup'].fillna('unknown')
combined['days_since_reg'] = (combined['auth_ts'] - combined['reg_ts']) // (60 * 60 * 24)

user_profile = combined.groupby('uid').agg(
    total_logins=('auth_ts', 'count'),
    first_login_day=('days_since_reg', 'min'),
    last_login_day=('days_since_reg', 'max'),
    total_revenue=('revenue', 'max'),
    testgroup=('testgroup', 'first'),
).reset_index()
user_profile['retention_span'] = user_profile['last_login_day'] - user_profile['first_login_day']
user_profile['is_payer'] = user_profile['total_revenue'] > 0

def categorize_spender(revenue):
    if revenue == 0:
        return '非付费用户'
    elif revenue <= 500:
        return '低消费 (≤$500)'
    elif revenue <= 5000:
        return '中消费 ($500-$5000)'
    else:
        return '高消费 (>$5000)'

tier_order = ['非付费用户', '低消费 (≤$500)', '中消费 ($500-$5000)', '高消费 (>$5000)']
user_profile['spending_tier'] = pd.Categorical(
    user_profile['total_revenue'].apply(categorize_spender), categories=tier_order, ordered=True)

tier_stats = user_profile.groupby('spending_tier', observed=False).agg(
    user_count=('uid', 'count'),
    avg_logins=('total_logins', 'mean'),
    pct_active=('total_logins', lambda x: (x > 1).mean() * 100),
    avg_retention_span=('retention_span', 'mean'),
).round(2)

print('消费层级 × 留存行为（倒 U 型验证）：')
print(tier_stats.to_string())

# 付费用户相关性（Spearman，检验是否线性）
payers = user_profile[user_profile['is_payer']]
rho_logins = spearmanr(payers['total_logins'], payers['total_revenue']).correlation
rho_span = spearmanr(payers['retention_span'], payers['total_revenue']).correlation
print(f'\n付费用户 Spearman 相关：登录次数 vs 消费 ρ={rho_logins:+.4f} | 留存跨度 vs 消费 ρ={rho_span:+.4f}')
print('结论：消费与活跃度无单调线性关系，呈倒 U 型——低消费层最活跃，高消费鲸鱼登录次数最低。')


消费层级 × 留存行为（倒 U 型验证）：
                  user_count  avg_logins  pct_active  avg_retention_span
spending_tier                                                           
非付费用户                 996665        9.57       23.84               34.13
低消费 (≤$500)             1606       21.08       23.85               80.43
中消费 ($500-$5000)        1612       15.60       24.57               58.14
高消费 (>$5000)             117       13.06       20.51               47.26

付费用户 Spearman 相关：登录次数 vs 消费 ρ=-0.0021 | 留存跨度 vs 消费 ρ=-0.0021
结论：消费与活跃度无单调线性关系，呈倒 U 型——低消费层最活跃，高消费鲸鱼登录次数最低。


In [22]:
# 5.5 消费层级可视化：登录次数 / 留存跨度（倒 U 型）
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
colors = ['#b0bec5', '#ffb74d', '#ff8a65', '#e57373']

bars1 = axes[0].bar(tier_stats.index, tier_stats['avg_logins'], color=colors, edgecolor='white')
axes[0].set_title('平均登录次数 (按消费层级)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('平均登录次数')
axes[0].set_xlabel('消费层级')
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars1, tier_stats['avg_logins']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{val:.1f}', ha='center', fontsize=11)

bars2 = axes[1].bar(tier_stats.index, tier_stats['avg_retention_span'], color=colors, edgecolor='white')
axes[1].set_title('平均留存跨度 / 天 (按消费层级)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('平均留存跨度 (天)')
axes[1].set_xlabel('消费层级')
axes[1].tick_params(axis='x', rotation=20)
for bar, val in zip(bars2, tier_stats['avg_retention_span']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, f'{val:.0f}', ha='center', fontsize=11)

box_data = [user_profile[user_profile['is_payer'] == False]['retention_span'],
            user_profile[user_profile['is_payer'] == True]['retention_span']]
bp = axes[2].boxplot(box_data, labels=['非付费用户', '付费用户'], patch_artist=True)
bp['boxes'][0].set_facecolor('#b0bec5')
bp['boxes'][1].set_facecolor('#e57373')
axes[2].set_title('留存跨度分布：付费 vs 非付费', fontsize=14, fontweight='bold')
axes[2].set_ylabel('留存跨度 (天)')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '05_spending_inverted_u.png'), dpi=150, bbox_inches='tight')
plt.show()


In [23]:
# 5.6 行为聚类：特征准备（三种聚类共用）
login_counts = auth_sorted.groupby('uid').size().reset_index(name='total_logins')
cluster_features = player_gap_summary[['uid', 'days_since_last_login', 'gap_std', 'lifecycle_days']].copy()
cluster_features = cluster_features.merge(login_counts, on='uid', how='left')
cluster_features = cluster_features.replace([np.inf, -np.inf], np.nan).dropna()

cluster_values = cluster_features[['days_since_last_login', 'gap_std', 'lifecycle_days', 'total_logins']]
cluster_scaler = StandardScaler()
X_cluster = cluster_scaler.fit_transform(cluster_values)

# 关键：silhouette_score 是 O(n²)，5 万采样会卡死，降采样到 8000 保证可运行
CLUSTER_SAMPLE = 8000
np.random.seed(42)
sample_idx = np.random.choice(len(X_cluster), min(CLUSTER_SAMPLE, len(X_cluster)), replace=False)
X_sample = X_cluster[sample_idx]

print(f'聚类特征矩阵: {cluster_features.shape[0]:,} 玩家 × {cluster_values.shape[1]} 特征')
print(f'轮廓系数计算采样规模: {len(X_sample):,}（规避 O(n²) 卡死）')
print('特征描述统计：')
print(cluster_values.describe().round(2).to_string())


聚类特征矩阵: 217,976 玩家 × 4 特征
轮廓系数计算采样规模: 8,000（规避 O(n²) 卡死）
特征描述统计：
       days_since_last_login    gap_std  lifecycle_days  total_logins
count              217976.00  217976.00       217976.00     217976.00
mean                    3.49       1.64          156.76         40.36
std                     0.71       0.49          378.24         94.62
min                     1.00       0.00            2.00          3.00
25%                     3.11       1.41           16.00          5.00
50%                     3.50       1.69           28.00          8.00
75%                     3.88       1.88           42.00         11.00
max                     6.00       3.54         7728.00       1929.00


In [24]:
# 5.7 K-Means：肘部法则 + 轮廓系数 + Davies-Bouldin 确定最优 K
k_range = range(2, 11)
inertias, sil_scores, db_scores = [], [], []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = km.fit_predict(X_sample)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_sample, labels))
    db_scores.append(davies_bouldin_score(X_sample, labels))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(list(k_range), inertias, 'bo-', markersize=8)
axes[0].set_title('K-Means 肘部法则', fontweight='bold')
axes[0].set_xlabel('聚类数 K'); axes[0].set_ylabel('惯性'); axes[0].grid(True, alpha=0.3)
axes[1].plot(list(k_range), sil_scores, 'go-', markersize=8)
axes[1].axhline(y=max(sil_scores), color='r', linestyle='--', alpha=0.5)
axes[1].set_title('轮廓系数 vs K 值', fontweight='bold')
axes[1].set_xlabel('聚类数 K'); axes[1].set_ylabel('轮廓系数'); axes[1].grid(True, alpha=0.3)
axes[2].plot(list(k_range), db_scores, 'ro-', markersize=8)
axes[2].set_title('Davies-Bouldin 指数 vs K 值（越低越好）', fontweight='bold')
axes[2].set_xlabel('聚类数 K'); axes[2].set_ylabel('DB 指数'); axes[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '05_kmeans_optimal_k.png'), dpi=150, bbox_inches='tight')
plt.show()

best_k = list(k_range)[np.argmax(sil_scores)]
print(f'推荐最优 K: {best_k}（轮廓系数 {max(sil_scores):.4f}）')


推荐最优 K: 2（轮廓系数 0.6043）


In [25]:
# 5.8 应用最优 K-Means 聚类 + PCA 可视化 + 聚类画像
final_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10, max_iter=300)
cluster_features['kmeans_cluster'] = final_kmeans.fit_predict(X_cluster)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cluster)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sc = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_features['kmeans_cluster'], cmap='viridis', alpha=0.4, s=4)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} 方差)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} 方差)')
axes[0].set_title(f'K-Means 聚类结果 (K={best_k}) - PCA 投影', fontweight='bold')
plt.colorbar(sc, ax=axes[0], label='聚类标签')

cluster_profile = cluster_features.groupby('kmeans_cluster')[
    ['days_since_last_login', 'gap_std', 'lifecycle_days', 'total_logins']].mean()
sns.heatmap(cluster_profile.T, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=axes[1], cbar_kws={'label': '特征均值'})
axes[1].set_title('各聚类行为特征热力图', fontweight='bold')
axes[1].set_xlabel('聚类'); axes[1].set_ylabel('特征')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '05_kmeans_clusters.png'), dpi=150, bbox_inches='tight')
plt.show()

print('各聚类特征均值：')
print(cluster_profile.round(2).to_string())
print('\n各聚类规模：')
print(cluster_features['kmeans_cluster'].value_counts().sort_index().to_string())


各聚类特征均值：
                days_since_last_login  gap_std  lifecycle_days  total_logins
kmeans_cluster                                                              
0                                3.49     1.63           64.77         17.35
1                                3.50     1.71         1272.02        319.40

各聚类规模：
kmeans_cluster
0    201366
1     16610


In [26]:
# 5.9 DBSCAN 密度聚类（参数扫描，识别噪声点/离群玩家）
eps_range = [0.3, 0.5, 0.8, 1.0, 1.5]
min_samples_range = [5, 10, 20]
best_dbscan, best_score = None, -1

for eps in eps_range:
    for ms in min_samples_range:
        labels = DBSCAN(eps=eps, min_samples=ms).fit_predict(X_sample)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = int((labels == -1).sum())
        if n_clusters >= 2:
            mask = labels != -1
            if mask.sum() > n_clusters * 2:
                score = silhouette_score(X_sample[mask], labels[mask])
                if score > best_score and n_clusters <= 10:
                    best_score, best_dbscan = score, {'eps': eps, 'min_samples': ms,
                                                       'labels': labels, 'n_clusters': n_clusters, 'n_noise': n_noise}

if best_dbscan:
    print(f'最佳 DBSCAN: eps={best_dbscan["eps"]}, min_samples={best_dbscan["min_samples"]}')
    print(f'  聚类数={best_dbscan["n_clusters"]}, 噪声点比例={best_dbscan["n_noise"]/len(X_sample)*100:.1f}%')
    print('  结论：大量噪声点表明玩家行为更接近连续谱系而非离散聚类。')
else:
    print('DBSCAN 未找到合适聚类结构（行为呈连续分布）。')


最佳 DBSCAN: eps=0.5, min_samples=20
  聚类数=7, 噪声点比例=0.8%
  结论：大量噪声点表明玩家行为更接近连续谱系而非离散聚类。


In [27]:
# 5.10 高斯混合模型 (GMM)：BIC/AIC + 轮廓系数选择组件数
gmm_range = range(2, 9)
gmm_bic, gmm_aic, gmm_sil = [], [], []
for nc in gmm_range:
    gmm = GaussianMixture(n_components=nc, covariance_type='full', random_state=42, n_init=3)
    labels = gmm.fit_predict(X_sample)
    gmm_bic.append(gmm.bic(X_sample))
    gmm_aic.append(gmm.aic(X_sample))
    gmm_sil.append(silhouette_score(X_sample, labels))

best_gmm_k = list(gmm_range)[np.argmin(gmm_bic)]
best_gmm = GaussianMixture(n_components=best_gmm_k, covariance_type='full', random_state=42, n_init=5)
gmm_labels_full = best_gmm.fit_predict(X_sample)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(list(gmm_range), gmm_bic, 'bo-', markersize=8, label='BIC')
axes[0].plot(list(gmm_range), gmm_aic, 'go-', markersize=8, label='AIC')
axes[0].set_title('GMM - BIC/AIC 信息准则', fontweight='bold'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(list(gmm_range), gmm_sil, 'ro-', markersize=8)
axes[1].set_title('GMM - 轮廓系数 vs 组件数', fontweight='bold'); axes[1].grid(True, alpha=0.3)
pca_gmm = PCA(n_components=2, random_state=42)
X_pca_gmm = pca_gmm.fit_transform(X_sample)
sc = axes[2].scatter(X_pca_gmm[:, 0], X_pca_gmm[:, 1], c=gmm_labels_full, cmap='viridis', alpha=0.5, s=5)
axes[2].set_title(f'GMM 聚类结果 (K={best_gmm_k})', fontweight='bold')
plt.colorbar(sc, ax=axes[2], label='组件')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '05_gmm.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'GMM 最佳组件数: {best_gmm_k}（BIC 最优）')


GMM 最佳组件数: 8（BIC 最优）


### 5.11 聚类方法对比总结

| 方法 | 优势 | 劣势 | 适用场景 |
|------|------|------|----------|
| K-Means | 速度快、易解释 | 需预设 K、球形假设 | 大规模数据初步分层 |
| DBSCAN | 无需预设 K、抗噪声 | 参数敏感、密度不均 | 发现异常/离群玩家 |
| GMM | 软分配、概率解释 | 需预设组件数、计算较重 | 玩家行为的概率建模 |

**核心洞察：** K-Means 识别的玩家原型可与手动分层交叉验证；DBSCAN 的大噪声比提示玩家行为是连续谱系；GMM 的软分配概率可用于定位「介于两种原型之间」的玩家，支撑精准干预。


## 6. 流失预测建模

基于第 5 章刻画的行为特征，构建监督式机器学习模型预测「早期流失」（生命周期 ≤ 7 天）：

- **逻辑回归**：可解释基线；
- **随机森林**：捕捉非线性与特征交互；
- **XGBoost**：梯度提升，表格数据通常最优。

> **口径说明：** 由于登录间隔特征（`days_since_last_login`）要求玩家至少发生 2 次登录，建模群体为「≥2 次登录」的约 21.8 万玩家，此子群体内早期流失率约 5.6%；而第 3 章报告的「全量早期流失率 79.4%」面向全部 100 万注册玩家（含仅 1 次登录即流失者）。两者口径不同，请勿混淆。


In [28]:
# 6.1 特征工程：从认证历史提取多维行为特征
player_features = auth_sorted.groupby('uid').agg(
    total_logins=('auth_ts', 'count'),
    first_login=('auth_date', 'min'),
    last_login=('auth_date', 'max'),
    mean_gap=('days_since_last_login', 'mean'),
    std_gap=('days_since_last_login', 'std'),
    max_gap=('days_since_last_login', 'max'),
    min_gap=('days_since_last_login', 'min'),
    median_gap=('days_since_last_login', 'median'),
).reset_index()

# 衍生特征
player_features['login_timespan_days'] = (player_features['last_login'] - player_features['first_login']).dt.days
player_features['active_days_ratio'] = player_features['total_logins'] / (player_features['login_timespan_days'] + 1)
# 仅登录一次的玩家缺失值填充
player_features['std_gap'] = player_features['std_gap'].fillna(0)
player_features['min_gap'] = player_features['min_gap'].fillna(0)
player_features['median_gap'] = player_features['median_gap'].fillna(player_features['mean_gap'])

# 合并生命周期与标签
player_features = player_features.merge(
    player_gap_summary[['uid', 'lifecycle_days', 'early_churn', 'risk_score', 'player_segment']],
    on='uid', how='inner')

print(f'特征矩阵形状: {player_features.shape}')
print(f'特征列: {list(player_features.columns)}')
print(f'目标（early_churn）分布:')
print(player_features['early_churn'].value_counts().to_string())
print(f'流失率: {player_features["early_churn"].mean()*100:.2f}%')


特征矩阵形状: (217976, 15)
特征列: ['uid', 'total_logins', 'first_login', 'last_login', 'mean_gap', 'std_gap', 'max_gap', 'min_gap', 'median_gap', 'login_timespan_days', 'active_days_ratio', 'lifecycle_days', 'early_churn', 'risk_score', 'player_segment']
目标（early_churn）分布:
early_churn
False    205848
True      12128
流失率: 5.56%


In [29]:
# 6.2 准备训练数据：标准化 + 分层划分
feature_cols = [
    'total_logins', 'mean_gap', 'std_gap', 'max_gap', 'min_gap',
    'median_gap', 'active_days_ratio', 'login_timespan_days',
    'lifecycle_days', 'risk_score'
]
X = player_features[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = player_features['early_churn'].astype(int)

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y)

print(f'训练集: {X_train.shape[0]:,} | 测试集: {X_test.shape[0]:,}')
print(f'训练集流失率: {y_train.mean()*100:.2f}% | 测试集流失率: {y_test.mean()*100:.2f}%')


训练集: 152,583 | 测试集: 65,393
训练集流失率: 5.56% | 测试集流失率: 5.56%


In [30]:
# 6.3 模型训练与评估：逻辑回归 + 随机森林（含 5 折交叉验证）
models = {
    '逻辑回归': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    '随机森林': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42,
                                       class_weight='balanced', n_jobs=-1),
}
results = {}
for name, model in models.items():
    print(f'\n训练模型: {name}')
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    print(f'  5 折交叉验证 ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'model': model,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_prob),
        'y_prob': y_prob, 'y_pred': y_pred,
    }
    print(f'  准确率 {results[name]["accuracy"]:.4f} | 精确率 {results[name]["precision"]:.4f} | '
          f'召回率 {results[name]["recall"]:.4f} | F1 {results[name]["f1"]:.4f} | ROC-AUC {results[name]["roc_auc"]:.4f}')



训练模型: 逻辑回归


  5 折交叉验证 ROC-AUC: 0.9983 (+/- 0.0004)


  准确率 0.9849 | 精确率 0.7861 | 召回率 1.0000 | F1 0.8802 | ROC-AUC 0.9985

训练模型: 随机森林


  5 折交叉验证 ROC-AUC: 1.0000 (+/- 0.0000)


  准确率 1.0000 | 精确率 1.0000 | 召回率 1.0000 | F1 1.0000 | ROC-AUC 1.0000


In [31]:
# 6.4 XGBoost 模型（梯度提升，注意 xgboost 3.x 已移除 use_label_encoder 参数）
try:
    from xgboost import XGBClassifier
    scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    xgb_model = XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        scale_pos_weight=scale_pos_weight, random_state=42
    )
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)
    y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
    results['XGBoost'] = {
        'model': xgb_model,
        'accuracy': accuracy_score(y_test, y_pred_xgb),
        'precision': precision_score(y_test, y_pred_xgb),
        'recall': recall_score(y_test, y_pred_xgb),
        'f1': f1_score(y_test, y_pred_xgb),
        'roc_auc': roc_auc_score(y_test, y_prob_xgb),
        'y_prob': y_prob_xgb, 'y_pred': y_pred_xgb,
    }
    print('XGBoost 结果: 准确率 %.4f | 精确率 %.4f | 召回率 %.4f | F1 %.4f | ROC-AUC %.4f'
          % (results['XGBoost']['accuracy'], results['XGBoost']['precision'],
             results['XGBoost']['recall'], results['XGBoost']['f1'], results['XGBoost']['roc_auc']))
except ImportError:
    print('XGBoost 未安装，跳过（可 pip install xgboost 后重跑）。')
except Exception as e:
    print(f'XGBoost 训练异常: {e}')

# 所有模型性能汇总
print('\n' + '=' * 60)
print('所有模型性能汇总')
print('=' * 60)
summary_rows = [{'模型': k, '准确率': f"{v['accuracy']:.4f}", '精确率': f"{v['precision']:.4f}",
                 '召回率': f"{v['recall']:.4f}", 'F1': f"{v['f1']:.4f}", 'ROC-AUC': f"{v['roc_auc']:.4f}"}
                for k, v in results.items()]
print(pd.DataFrame(summary_rows).to_string(index=False))


XGBoost 结果: 准确率 1.0000 | 精确率 1.0000 | 召回率 1.0000 | F1 1.0000 | ROC-AUC 1.0000

所有模型性能汇总
     模型    准确率    精确率    召回率     F1 ROC-AUC
   逻辑回归 0.9849 0.7861 1.0000 0.8802  0.9985
   随机森林 1.0000 1.0000 1.0000 1.0000  1.0000
XGBoost 1.0000 1.0000 1.0000 1.0000  1.0000


In [32]:
# 6.5 模型对比可视化：ROC 曲线 + 指标对比 + 混淆矩阵
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[0].plot(fpr, tpr, lw=2, label=f"{name} (AUC={res['roc_auc']:.3f})")
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='随机猜测')
axes[0].set_xlabel('假阳性率'); axes[0].set_ylabel('真阳性率')
axes[0].set_title('ROC 曲线对比', fontweight='bold'); axes[0].legend(loc='lower right'); axes[0].grid(True, alpha=0.3)

metric_names = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
x = np.arange(len(metric_names))
width = 0.25
for i, (name, res) in enumerate(results.items()):
    vals = [res[m] for m in metric_names]
    axes[1].bar(x + i * width, vals, width, label=name, alpha=0.85)
axes[1].set_xticks(x + width)
axes[1].set_xticklabels(['准确率', '精确率', '召回率', 'F1', 'ROC-AUC'])
axes[1].set_ylim(0, 1.1); axes[1].set_title('模型性能指标对比', fontweight='bold'); axes[1].legend()

best_name = max(results, key=lambda k: results[k]['roc_auc'])
cm = confusion_matrix(y_test, results[best_name]['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2], xticklabels=['留存', '流失'], yticklabels=['留存', '流失'])
axes[2].set_xlabel('预测标签'); axes[2].set_ylabel('真实标签')
axes[2].set_title(f'{best_name} - 混淆矩阵', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '06_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'最佳模型: {best_name}（ROC-AUC {results[best_name]["roc_auc"]:.4f}）')


最佳模型: 随机森林（ROC-AUC 1.0000）


In [33]:
# 6.6 特征重要性：随机森林 + 逻辑回归
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rf_model = results['随机森林']['model']
rf_imp = rf_model.feature_importances_
rf_idx = np.argsort(rf_imp)[::-1]
axes[0].barh(range(len(feature_cols)), rf_imp[rf_idx], color=plt.cm.viridis(np.linspace(0.2, 0.9, len(feature_cols))))
axes[0].set_yticks(range(len(feature_cols)))
axes[0].set_yticklabels([feature_cols[i] for i in rf_idx])
axes[0].set_title('随机森林 - 特征重要性', fontweight='bold')
axes[0].invert_yaxis()

lr_model = results['逻辑回归']['model']
lr_coef = np.abs(lr_model.coef_[0])
lr_idx = np.argsort(lr_coef)[::-1]
axes[1].barh(range(len(feature_cols)), lr_coef[lr_idx], color=plt.cm.viridis(np.linspace(0.2, 0.9, len(feature_cols))))
axes[1].set_yticks(range(len(feature_cols)))
axes[1].set_yticklabels([feature_cols[i] for i in lr_idx])
axes[1].set_title('逻辑回归 - |系数| 重要性', fontweight='bold')
axes[1].invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '06_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

importance_df = pd.DataFrame({'特征': feature_cols, '随机森林重要性': rf_imp, '逻辑回归|系数|': np.abs(lr_model.coef_[0])})
importance_df['平均排名'] = (importance_df['随机森林重要性'].rank(ascending=False) +
                             importance_df['逻辑回归|系数|'].rank(ascending=False)) / 2
print(importance_df.sort_values('平均排名').to_string(index=False))


                 特征  随机森林重要性  逻辑回归|系数|  平均排名
            max_gap 0.033947 18.980832   3.0
  active_days_ratio 0.204354  7.696174   3.5
         risk_score 0.007467  9.659877   5.0
login_timespan_days 0.366041  2.626790   5.5
     lifecycle_days 0.237610  2.626790   5.5
           mean_gap 0.022945  7.098830   5.5
            min_gap 0.001085  8.876531   6.0
       total_logins 0.116676  5.561806   6.0
         median_gap 0.009264  6.788466   6.5
            std_gap 0.000610  6.243845   8.5


## 7. 时间序列与活动趋势

将聚合快照视角扩展为动态行为时间线，分析全局活跃趋势、个体生命周期曲线、流失前行为前兆与季节性模式。


In [34]:
# 7.1 全局周度活动时间序列 + 移动平均
auth_sorted['auth_week'] = auth_sorted['auth_date'].dt.to_period('W').dt.start_time
weekly_logins = auth_sorted.groupby('auth_week').size().reset_index(name='login_count')
weekly_active = auth_sorted.groupby('auth_week')['uid'].nunique().reset_index(name='active_users')
weekly_logins = weekly_logins.merge(weekly_active, on='auth_week')

reg_week = reg_data.copy()
reg_week['reg_week'] = reg_week['reg_date'].dt.to_period('W').dt.start_time
weekly_reg = reg_week.groupby('reg_week')['uid'].nunique().reset_index(name='new_users')

fig, axes = plt.subplots(2, 1, figsize=(18, 10))
axes[0].fill_between(weekly_logins['auth_week'], weekly_logins['login_count'], alpha=0.3, color='#2196F3')
axes[0].plot(weekly_logins['auth_week'], weekly_logins['login_count'], color='#2196F3', linewidth=1)
axes[0].plot(weekly_logins['auth_week'], weekly_logins['login_count'].rolling(12, center=True).mean(),
             color='red', linewidth=2, label='12周移动平均')
axes[0].set_title('每周总登录量趋势', fontweight='bold')
axes[0].set_ylabel('登录次数'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].fill_between(weekly_reg['reg_week'], weekly_reg['new_users'], alpha=0.3, color='#4CAF50')
axes[1].plot(weekly_reg['reg_week'], weekly_reg['new_users'], color='#4CAF50', linewidth=1)
axes[1].plot(weekly_reg['reg_week'], weekly_reg['new_users'].rolling(12, center=True).mean(),
             color='red', linewidth=2, label='12周移动平均')
axes[1].set_title('每周新注册量趋势', fontweight='bold')
axes[1].set_xlabel('周'); axes[1].set_ylabel('新注册用户数'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '07_global_weekly_trends.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'时间序列范围: {weekly_logins["auth_week"].min().date()} 至 {weekly_logins["auth_week"].max().date()}')
print(f'周均登录量: {weekly_logins["login_count"].mean():.0f} | 周均活跃用户: {weekly_logins["active_users"].mean():.0f}')


时间序列范围: 1998-11-16 至 2020-09-21
周均登录量: 8673 | 周均活跃用户: 5341


In [35]:
# 7.2 个体玩家活动生命周期曲线（按生命周期层级抽样）
from matplotlib.ticker import MaxNLocator

lifecycle_bins = [0, 7, 30, 90, 365, float('inf')]
lifecycle_labels = ['极短(<7天)', '短(7-30天)', '中(30-90天)', '长(90-365天)', '极长(>365天)']
player_gap_summary['lifecycle_tier'] = pd.cut(player_gap_summary['lifecycle_days'],
                                              bins=lifecycle_bins, labels=lifecycle_labels)

np.random.seed(42)
fig, axes = plt.subplots(len(lifecycle_labels), 1, figsize=(16, 3 * len(lifecycle_labels)))
for idx, tier in enumerate(lifecycle_labels):
    tier_uids = player_gap_summary[player_gap_summary['lifecycle_tier'] == tier]['uid'].values
    if len(tier_uids) == 0:
        axes[idx].set_title(f'{tier} - 无玩家'); continue
    sample_uids = np.random.choice(tier_uids, min(3, len(tier_uids)), replace=False)
    for uid in sample_uids:
        pauth = auth_sorted[auth_sorted['uid'] == uid].sort_values('auth_date')
        reg_date = reg_data[reg_data['uid'] == uid]['reg_date'].values[0]
        relative_days = (pauth['auth_date'] - reg_date).dt.days
        weekly_activity = (relative_days // 7).value_counts().sort_index()
        axes[idx].plot(weekly_activity.index, weekly_activity.values, marker='.', markersize=3, alpha=0.7, linewidth=1, label=f'玩家 {uid}')
    axes[idx].set_xlabel('相对周数 (注册后)')
    axes[idx].set_ylabel('周登录次数')
    axes[idx].set_title(f'生命周期层级: {tier} (玩家数: {len(tier_uids):,})')
    axes[idx].legend(fontsize=7, loc='upper right'); axes[idx].grid(True, alpha=0.3)
    axes[idx].xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '07_player_lifecycles.png'), dpi=150, bbox_inches='tight')
plt.show()
print('生命周期层级分布：')
print(player_gap_summary['lifecycle_tier'].value_counts().to_string())


生命周期层级分布：
lifecycle_tier
短(7-30天)      110608
中(30-90天)      52173
极长(>365天)      27402
长(90-365天)     15665
极短(<7天)        12128


In [36]:
# 7.3 流失前行为模式：流失 vs 留存玩家的活动衰减对比
churned_uids = player_gap_summary[player_gap_summary['early_churn']]['uid'].values
retained_uids = player_gap_summary[~player_gap_summary['early_churn']]['uid'].sample(
    n=min(1000, (~player_gap_summary['early_churn']).sum()), random_state=42).values

def compute_weekly_profile(uid_list, label):
    all_weeks = []
    for uid in uid_list:
        pauth = auth_sorted[auth_sorted['uid'] == uid].sort_values('auth_date')
        if len(pauth) < 2:
            continue
        reg_date = reg_data[reg_data['uid'] == uid]['reg_date'].values[0]
        last_date = pauth['auth_date'].max()
        lifecycle = max((last_date - reg_date).days, 1)
        rel_week = ((pauth['auth_date'] - reg_date).dt.days // 7).value_counts()
        for week, cnt in rel_week.items():
            all_weeks.append({'normalized_week': week / max(lifecycle / 7, 1) * 100,
                              'login_count': cnt, 'type': label})
    return pd.DataFrame(all_weeks)

churn_profile = compute_weekly_profile(churned_uids[:500], '流失玩家')
retain_profile = compute_weekly_profile(retained_uids, '留存玩家')
activity_profile = pd.concat([churn_profile, retain_profile], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label, color in [('流失玩家', 'red'), ('留存玩家', 'green')]:
    sub = activity_profile[activity_profile['type'] == label]
    binned = sub.groupby(pd.cut(sub['normalized_week'], bins=20))['login_count'].mean()
    centers = [(b.left + b.right) / 2 for b in binned.index]
    axes[0].plot(centers, binned.values, color=color, linewidth=2, label=label, marker='o', markersize=4)
axes[0].set_xlabel('归一化生命周期进度 (%)'); axes[0].set_ylabel('平均周登录次数')
axes[0].set_title('流失 vs 留存玩家活动曲线', fontweight='bold'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

decay_stats = {}
for label in ['流失玩家', '留存玩家']:
    sub = activity_profile[activity_profile['type'] == label]
    early_act = sub[sub['normalized_week'] <= 50]['login_count'].mean()
    late_act = sub[sub['normalized_week'] > 50]['login_count'].mean()
    decay = (early_act - late_act) / max(early_act, 0.001) * 100
    decay_stats[label] = (early_act, late_act, decay)
for label, color in [('流失玩家', 'red'), ('留存玩家', 'green')]:
    e, l, _ = decay_stats[label]
    axes[1].bar(label, e, color=color, alpha=0.6, label=f'{label} 前期')
    axes[1].bar(label, l, color=color, alpha=0.3, hatch='//', label=f'{label} 后期')
axes[1].set_ylabel('平均周登录次数'); axes[1].set_title('生命周期前后半段活动对比', fontweight='bold')
handles, labels = axes[1].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
axes[1].legend(by_label.values(), by_label.keys(), fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '07_churn_precursors.png'), dpi=150, bbox_inches='tight')
plt.show()

print('流失预警信号分析：')
for label in ['流失玩家', '留存玩家']:
    e, l, d = decay_stats[label]
    print(f'  {label}: 前期={e:.2f} 次/周, 后期={l:.2f} 次/周, 衰减率={d:.1f}%')


流失预警信号分析：
  流失玩家: 前期=2.82 次/周, 后期=1.00 次/周, 衰减率=64.6%
  留存玩家: 前期=1.80 次/周, 后期=1.72 次/周, 衰减率=4.5%


In [37]:
# 7.4 月度活动模式与季节性
monthly_logins = auth_sorted.groupby(auth_sorted['auth_date'].dt.to_period('M').dt.start_time).agg(
    total_logins=('auth_ts', 'count'),
    unique_users=('uid', 'nunique'),
    avg_per_user=('uid', lambda x: len(x) / x.nunique())
).reset_index()
monthly_logins.columns = ['auth_month', 'total_logins', 'unique_users', 'avg_per_user']
monthly_logins['year'] = monthly_logins['auth_month'].dt.year
monthly_logins['month'] = monthly_logins['auth_month'].dt.month

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes[0, 0].fill_between(monthly_logins['auth_month'], monthly_logins['total_logins'], alpha=0.4, color='steelblue')
axes[0, 0].plot(monthly_logins['auth_month'], monthly_logins['total_logins'], color='steelblue', linewidth=1.5)
axes[0, 0].set_title('月度总登录量趋势', fontweight='bold'); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].bar(monthly_logins['auth_month'], monthly_logins['avg_per_user'], color='seagreen', alpha=0.7, width=20)
axes[0, 1].set_title('月度人均登录频率', fontweight='bold'); axes[0, 1].grid(True, alpha=0.3, axis='y')

seasonal_pivot = monthly_logins.pivot_table(values='total_logins', index='year', columns='month', aggfunc='sum')
sns.heatmap(seasonal_pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[1, 0], cbar_kws={'label': '总登录量'})
axes[1, 0].set_title('年度-月度登录量热力图', fontweight='bold')

monthly_by_month = monthly_logins.groupby('month')['total_logins'].apply(list)
monthly_data = [monthly_by_month.get(m, []) for m in range(1, 13)]
bp = axes[1, 1].boxplot(monthly_data, labels=range(1, 13), patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('steelblue'); patch.set_alpha(0.6)
axes[1, 1].set_title('各月份登录量分布（跨年汇总）', fontweight='bold'); axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '07_monthly_patterns.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'最高活动月份: {monthly_logins.loc[monthly_logins["total_logins"].idxmax(), "auth_month"].strftime("%Y-%m")}')
print(f'月均登录量: {monthly_logins["total_logins"].mean():.0f} | 月均人均登录: {monthly_logins["avg_per_user"].mean():.2f}')


最高活动月份: 2020-08
月均登录量: 37504 | 月均人均登录: 4.46


## 8. 游戏主题活动评估框架（任务 3）

针对 "Plants & Gardens" 游戏的限时主题活动，构建一套可复用的评估指标体系。当活动机制复杂化（例如「通关失败回退关卡」）时，评估口径需要相应调整。


In [38]:
# 8.1 活动评估七维指标体系（框架化落地）
activity_framework = pd.DataFrame([
    ['玩家活动', '活动参与人数', '开始参与活动的独特玩家总数'],
    ['玩家活动', '活动完成率', '完成活动的参与者 / 开始总人数'],
    ['参与深度', '平均参与时间', '从开始到完成的平均时长'],
    ['参与深度', '通过关卡数', '活动期间平均通过关卡数'],
    ['成功率', '成功尝试比例', '成功完成尝试 / 总尝试次数'],
    ['成功率', '平均尝试次数', '成功前平均尝试次数（难度感知）'],
    ['奖励获取', '奖励数量 / 价值', '获得独特物品或奖励的总价值'],
    ['财务指标', '活动 ARPU / 总收入', '活动参与者的平均收入与总收入'],
    ['行为指标', '活动后留存率', '活动后 7/14/30 天返回比例'],
    ['行为指标', '会话频率变化', '活动前后游戏会话频率差异'],
    ['满意度', '评分与评论', '玩家主观反馈与评分'],
], columns=['维度', '指标', '定义/口径'])
print(activity_framework.to_string(index=False))

print('\n机制复杂化（失败回退关卡）后的指标变化：')
print('  成功尝试比例 ↓（难度上升） | 平均尝试次数 ↑（需更多尝试） | 平均参与时间 ↑')
print('  ARPU ↑/↓（难度或驱动/抑制付费） | 满意度分化（挑战型 vs 休闲型）')


  维度            指标             定义/口径
玩家活动        活动参与人数     开始参与活动的独特玩家总数
玩家活动         活动完成率  完成活动的参与者 / 开始总人数
参与深度        平均参与时间       从开始到完成的平均时长
参与深度         通过关卡数       活动期间平均通过关卡数
 成功率        成功尝试比例    成功完成尝试 / 总尝试次数
 成功率        平均尝试次数   成功前平均尝试次数（难度感知）
奖励获取     奖励数量 / 价值     获得独特物品或奖励的总价值
财务指标 活动 ARPU / 总收入    活动参与者的平均收入与总收入
行为指标        活动后留存率 活动后 7/14/30 天返回比例
行为指标        会话频率变化      活动前后游戏会话频率差异
 满意度         评分与评论         玩家主观反馈与评分

机制复杂化（失败回退关卡）后的指标变化：
  成功尝试比例 ↓（难度上升） | 平均尝试次数 ↑（需更多尝试） | 平均参与时间 ↑
  ARPU ↑/↓（难度或驱动/抑制付费） | 满意度分化（挑战型 vs 休闲型）


## 9. 实时监控预警系统

将离线行为分析升级为可落地的运营监控框架：基于滑动时间窗口计算**动态风险评分**，按四级阈值（绿/黄/橙/红）分级，自动生成预警列表与干预策略，并以仪表板形式呈现关键运营指标。


In [39]:
# 9.1 动态风险评分引擎：滑动时间窗口滚动指标
auth_sorted['time_window'] = auth_sorted['auth_date'].dt.to_period('M')
time_windows = sorted(auth_sorted['time_window'].unique())
print(f'总时间窗口数: {len(time_windows)} | 范围: {time_windows[0]} 至 {time_windows[-1]}')

def compute_rolling_metrics(auth_df, reg_df, window_end):
    '''计算截至指定窗口的滚动行为指标。'''
    hist_data = auth_df[auth_df['auth_date'] <= window_end].copy()
    if len(hist_data) == 0:
        return pd.DataFrame()
    metrics = hist_data.groupby('uid').agg(
        累计登录次数=('auth_ts', 'count'),
        最后登录日期=('auth_date', 'max'),
        首次登录日期=('auth_date', 'min'),
        平均登录间隔=('days_since_last_login', 'mean'),
        登录间隔标准差=('days_since_last_login', 'std'),
    ).reset_index()
    metrics = metrics.merge(reg_df[['uid', 'reg_date']], on='uid', how='left')
    metrics['生命周期天数'] = (metrics['最后登录日期'] - metrics['reg_date']).dt.days
    metrics['距上次登录天数'] = (window_end - metrics['最后登录日期']).dt.days
    metrics['活跃密度'] = metrics['累计登录次数'] / (metrics['生命周期天数'] + 1)
    metrics['平均登录间隔'] = metrics['平均登录间隔'].fillna(7)
    metrics['登录间隔标准差'] = metrics['登录间隔标准差'].fillna(0)
    return metrics

sample_windows = [
    time_windows[len(time_windows) // 4],
    time_windows[len(time_windows) // 2],
    time_windows[len(time_windows) * 3 // 4],
]
rolling_metrics_list = []
for i, w in enumerate(sample_windows):
    metrics = compute_rolling_metrics(auth_sorted, reg_data, w.end_time)
    metrics['窗口标签'] = f'T{i+1}: {w}'
    rolling_metrics_list.append(metrics)
    print(f'  窗口 {metrics["窗口标签"].iloc[0]}: {len(metrics):,} 玩家')


总时间窗口数: 256 | 范围: 1998-11 至 2020-09
  窗口 T1: 2004-10: 70 玩家
  窗口 T2: 2010-02: 1,751 玩家
  窗口 T3: 2015-06: 43,134 玩家


In [40]:
# 9.2 动态风险评分与四级风险等级
def calculate_dynamic_risk(metrics_df):
    '''计算 0-100 动态风险评分与绿/黄/橙/红等级。'''
    df = metrics_df.copy()
    for col in ['平均登录间隔', '登录间隔标准差', '距上次登录天数']:
        q99 = df[col].quantile(0.99)
        df[col + '_norm'] = np.clip(df[col] / max(q99, 0.01), 0, 1)
    for col in ['累计登录次数', '生命周期天数', '活跃密度']:
        q99 = df[col].quantile(0.99)
        df[col + '_norm'] = 1 - np.clip(df[col] / max(q99, 0.01), 0, 1)
    df['动态风险评分'] = (
        df['平均登录间隔_norm'] * 0.20 + df['登录间隔标准差_norm'] * 0.20 +
        df['距上次登录天数_norm'] * 0.25 + df['累计登录次数_norm'] * 0.10 +
        df['生命周期天数_norm'] * 0.10 + df['活跃密度_norm'] * 0.15) * 100
    df['风险等级'] = pd.cut(df['动态风险评分'], bins=[-1, 25, 50, 75, 101],
                            labels=['绿色-安全', '黄色-关注', '橙色-警告', '红色-高危'])
    return df

for i in range(len(rolling_metrics_list)):
    rolling_metrics_list[i] = calculate_dynamic_risk(rolling_metrics_list[i])

print('各时间窗口风险等级分布：')
for metrics in rolling_metrics_list:
    label = metrics['窗口标签'].iloc[0]
    dist = metrics['风险等级'].value_counts()
    total = len(metrics)
    line = f'  {label}: ' + ' | '.join(f'{k} {v:,}({v/total*100:.1f}%)' for k, v in dist.items())
    print(line)
    high = (metrics['动态风险评分'] >= 75).sum()
    print(f'    → 高危玩家: {high:,} ({high/total*100:.1f}%)')


各时间窗口风险等级分布：
  T1: 2004-10: 黄色-关注 38(54.3%) | 橙色-警告 32(45.7%) | 绿色-安全 0(0.0%) | 红色-高危 0(0.0%)
    → 高危玩家: 0 (0.0%)
  T2: 2010-02: 黄色-关注 1,226(70.0%) | 橙色-警告 516(29.5%) | 红色-高危 7(0.4%) | 绿色-安全 2(0.1%)
    → 高危玩家: 7 (0.4%)
  T3: 2015-06: 黄色-关注 30,366(70.4%) | 橙色-警告 12,519(29.0%) | 红色-高危 187(0.4%) | 绿色-安全 62(0.1%)
    → 高危玩家: 187 (0.4%)


In [41]:
# 9.3 预警触发系统 + 自动化响应策略
current_metrics = rolling_metrics_list[-1].copy()

def generate_alerts(metrics_df):
    alerts = []
    red = metrics_df[metrics_df['风险等级'] == '红色-高危']
    if len(red) > 0:
        alerts.append(['P0-紧急', '高危玩家检测', f'{len(red):,} 名玩家处于红色高危状态',
                       '立即推送个性化召回活动 + 客服1对1联系', len(red)])
    orange = metrics_df[metrics_df['风险等级'] == '橙色-警告']
    if len(orange) > 0:
        alerts.append(['P1-重要', '警告玩家检测', f'{len(orange):,} 名玩家处于橙色警告状态',
                       '推送定向奖励通知 + 游戏内活动邀请', len(orange)])
    inactive_7d = metrics_df[metrics_df['距上次登录天数'] >= 7]
    if len(inactive_7d) > 0:
        alerts.append(['P1-重要', '长期不活跃检测', f'{len(inactive_7d):,} 名玩家 7 天以上未登录',
                       '发送回归奖励推送', len(inactive_7d)])
    low_density = metrics_df[metrics_df['活跃密度'] < metrics_df['活跃密度'].quantile(0.1)]
    if len(low_density) > 0:
        alerts.append(['P2-观察', '低活跃密度检测', f'{len(low_density):,} 名玩家活跃密度处于最低 10%',
                       '纳入观察列表 + 分析行为模式', len(low_density)])
    return pd.DataFrame(alerts, columns=['优先级', '规则', '描述', '建议操作', '影响玩家数'])

alerts_df = generate_alerts(current_metrics)
print('当前预警列表：')
print(alerts_df.to_string(index=False))

print('\n自动化响应策略矩阵：')
print('  绿色-安全 (<25) → 常规内容推送 | 黄色-关注 (25-50) → 个性化推荐')
print('  橙色-警告 (50-75) → 定向奖励+限时活动 | 红色-高危 (≥75) → 客服1对1+回归礼包')


当前预警列表：
  优先级      规则                    描述                  建议操作  影响玩家数
P0-紧急  高危玩家检测       187 名玩家处于红色高危状态 立即推送个性化召回活动 + 客服1对1联系    187
P1-重要  警告玩家检测    12,519 名玩家处于橙色警告状态    推送定向奖励通知 + 游戏内活动邀请  12519
P1-重要 长期不活跃检测   40,187 名玩家 7 天以上未登录              发送回归奖励推送  40187
P2-观察 低活跃密度检测 4,248 名玩家活跃密度处于最低 10%       纳入观察列表 + 分析行为模式   4248

自动化响应策略矩阵：
  绿色-安全 (<25) → 常规内容推送 | 黄色-关注 (25-50) → 个性化推荐
  橙色-警告 (50-75) → 定向奖励+限时活动 | 红色-高危 (≥75) → 客服1对1+回归礼包


In [42]:
# 9.4 运营监控仪表板
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)
fig.suptitle('玩家留存实时监控仪表板', fontsize=18, fontweight='bold', y=0.98)

# 1 风险评分分布
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(current_metrics['动态风险评分'], bins=50, color='steelblue', alpha=0.7, edgecolor='white')
for threshold, color, label in [(25, '#2ecc71', '安全线'), (50, '#f1c40f', '关注线'),
                                (75, '#e67e22', '警告线'), (90, '#e74c3c', '高危线')]:
    ax1.axvline(x=threshold, color=color, linestyle='--', linewidth=1, alpha=0.7, label=label)
ax1.set_title('风险评分分布', fontweight='bold'); ax1.legend(fontsize=7); ax1.grid(True, alpha=0.3)

# 2 风险等级饼图
ax2 = fig.add_subplot(gs[0, 1])
risk_counts = current_metrics['风险等级'].value_counts()
level_order = ['绿色-安全', '黄色-关注', '橙色-警告', '红色-高危']
colors_risk = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
ax2.pie([risk_counts.get(l, 0) for l in level_order], labels=level_order, colors=colors_risk,
        autopct='%1.1f%%', startangle=90, textprops={'fontsize': 8})
ax2.set_title('当前风险等级构成', fontweight='bold')

# 3 关键指标面板
ax3 = fig.add_subplot(gs[0, 2]); ax3.axis('off')
total = len(current_metrics)
high_risk = (current_metrics['动态风险评分'] >= 75).sum()
inactive_7d = (current_metrics['距上次登录天数'] >= 7).sum()
metrics_text = (f'总监控玩家: {total:,}\n平均风险评分: {current_metrics["动态风险评分"].mean():.1f}\n'
                f'高危玩家(≥75): {high_risk:,} ({high_risk/total*100:.1f}%)\n'
                f'7天不活跃: {inactive_7d:,} ({inactive_7d/total*100:.1f}%)\n'
                f'更新时间: {time_windows[-1]}')
ax3.text(0.05, 0.5, metrics_text, transform=ax3.transAxes, fontsize=12, verticalalignment='center',
         fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='#f8f9fa', alpha=0.8))
ax3.set_title('关键指标面板', fontweight='bold')

# 4 各窗口风险趋势
ax4 = fig.add_subplot(gs[1, :])
window_labels = [m['窗口标签'].iloc[0] for m in rolling_metrics_list]
mean_risks = [m['动态风险评分'].mean() for m in rolling_metrics_list]
high_pcts = [(m['动态风险评分'] >= 75).mean() * 100 for m in rolling_metrics_list]
ax4.plot(range(len(window_labels)), mean_risks, 'bo-', linewidth=2, markersize=10, label='平均风险评分')
ax4_twin = ax4.twinx()
ax4_twin.plot(range(len(window_labels)), high_pcts, 'rs-', linewidth=2, markersize=10, label='高危比例 (%)')
ax4.set_xticks(range(len(window_labels))); ax4.set_xticklabels(window_labels, fontsize=9)
ax4.set_ylabel('平均风险评分'); ax4_twin.set_ylabel('高危比例 (%)'); ax4.set_title('风险指标时间趋势', fontweight='bold')
ax4.legend(loc='upper left'); ax4.grid(True, alpha=0.3)

# 5 不活跃天数分布
ax5 = fig.add_subplot(gs[2, 0])
ax5.hist(current_metrics['距上次登录天数'].clip(upper=30), bins=30, color='coral', alpha=0.7, edgecolor='white')
ax5.set_title('不活跃天数分布 (上限30天)', fontweight='bold'); ax5.grid(True, alpha=0.3)

# 6 活跃密度分布
ax6 = fig.add_subplot(gs[2, 1])
ax6.hist(current_metrics['活跃密度'].clip(upper=current_metrics['活跃密度'].quantile(0.95)),
         bins=30, color='seagreen', alpha=0.7, edgecolor='white')
ax6.set_title('活跃密度分布', fontweight='bold'); ax6.grid(True, alpha=0.3)

# 7 风险因素权重
ax7 = fig.add_subplot(gs[2, 2])
factors = ['距上次登录', '登录间隔', '间隔稳定性', '活跃密度', '累计登录', '生命周期']
weights = [0.25, 0.20, 0.20, 0.15, 0.10, 0.10]
bars = ax7.barh(factors, weights, color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(factors))), alpha=0.8)
for bar, w in zip(bars, weights):
    ax7.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, f'{w:.0%}', va='center', fontsize=9)
ax7.set_xlim(0, 0.35); ax7.set_title('风险评分因子权重', fontweight='bold'); ax7.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '09_monitoring_dashboard.png'), dpi=150, bbox_inches='tight')
plt.show()
print('监控仪表板已生成。')


监控仪表板已生成。


## 10. 综合结论与业务建议

### 1. 留存洞察
- **首月留存率 15.4%~18.9%**（2015-2020 群组），注册后前 5 天是关键干预窗口（D1 留存 4.0%，D5 升至 6.4%）；
- **早期流失率 79.4%**（生命周期 ≤7 天），其中 76.5% 的用户注册后仅 1 天即流失。

### 2. A/B 测试结论
- 测试组 ARPPU **+12.75%**（$2,664 → $3,004，p<0.001），CR 仅微降 0.06pp（0.954% → 0.891%），**建议采用测试组促销方案**。

### 3. 玩家画像洞察
- 消费-留存呈**倒 U 型**：低消费层（≤$500）平均登录 21.1 次、留存跨度 80.4 天（最高）；高消费鲸鱼（>$5000）登录 13.1 次（付费用户中最低）。

### 4. 业务建议

| 优先级 | 建议 | 预期影响 |
|--------|------|----------|
| 🔴 高 | 优化注册后前 5 天新手体验 | 降低早期流失率 |
| 🔴 高 | 采用测试组 (b) 促销方案 | 提升 ARPPU |
| 🟡 中 | 按行为聚类差异化运营 | 提升整体参与度 |
| 🟡 中 | 部署流失预测模型 + 实时预警 | 前置干预高风险玩家 |
| 🟢 低 | 建立实时监控仪表板 | 提升运营效率 |
